In [44]:
# Cell 1: Install dependencies
!pip install -q pymupdf opencv-python-headless pillow pandas tqdm transformers torch torchvision open_clip_torch
!pip install -q paddlepaddle paddleocr "paddlex[ocr]==3.7.2"


In [45]:
# Cell 2: Imports
import os
import re
import json
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pymupdf
import torch
import open_clip
from PIL import Image
from tqdm import tqdm
from IPython.display import display


In [46]:
# Cell 3: Configuration for Kaggle
PDF_PATH = "/kaggle/input/datasets/boopaland3v/sample-pdf-data/The_European_Medical_Device_Regulation-SECURED.pdf"

WORK_DIR = Path("/kaggle/working/flowchart_detection_v6")
EMBEDDED_DIR = WORK_DIR / "01_embedded_images"
RENDERED_DIR = WORK_DIR / "02_rendered_vector_pages"
TABLE_DEBUG_DIR = WORK_DIR / "03_table_debug"

# Rendering / filtering
RENDER_SCALE = 2
MIN_DRAWINGS_TO_RENDER = 20
MIN_WIDTH = 250
MIN_HEIGHT = 180
MIN_AREA = 60_000

# Performance controls
MAX_LAYOUT_INPUTS = 160        # Paddle layout is cheaper than OpenCLIP. Keep this reasonably high.
MAX_OPENCLIP_INPUTS = 100      # OpenCLIP should see only the stronger survivors.
MAX_PADDLEOCR_VL_ITEMS = 20    # PaddleOCR-VL is heavy. Use only for uncertain cases.
USE_PADDLE_LAYOUT = True
USE_PADDLEOCR_VL = True

OUTPUT_FOLDERS = [EMBEDDED_DIR, RENDERED_DIR, TABLE_DEBUG_DIR]
for folder in OUTPUT_FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)

print("PDF path:", PDF_PATH)
print("Work dir:", WORK_DIR)


PDF path: /kaggle/input/datasets/boopaland3v/sample-pdf-data/The_European_Medical_Device_Regulation-SECURED.pdf
Work dir: /kaggle/working/flowchart_detection_v6


In [47]:
# Cell 4: Clean workspace for every full run
for folder in OUTPUT_FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)
    for item in folder.iterdir():
        if item.is_file():
            item.unlink()
        elif item.is_dir():
            shutil.rmtree(item)

print("Workspace cleaned.")
for folder in OUTPUT_FOLDERS:
    print(f"{folder.name:30s}: {len(list(folder.iterdir()))} files")


Workspace cleaned.
01_embedded_images            : 0 files
02_rendered_vector_pages      : 0 files
03_table_debug                : 0 files


In [48]:
# Cell 5: Utility helpers

FLOW_WORDS = [
    "flowchart", "flow chart", "workflow", "process", "procedure", "decision",
    "start", "end", "yes", "no", "define", "manufacturer flow", "device flow",
    "submit", "approval", "classify", "covered", "excluded", "continue"
]

TABLE_WORDS = [
    "table", "reference map", "article numbers", "annex numbers", "key:",
    "row", "column", "contents", "index", "list of tables", "chapter no",
    "this part is referred to by", "this part also refers to"
]

NEGATIVE_VISUAL_WORDS = [
    "logo", "cover", "guidebook", "series", "regulatory affairs professionals society",
    "meddev solutions", "reference map"
]


def safe_image_size(path):
    try:
        img = Image.open(path)
        w, h = img.size
        return int(w), int(h), int(w * h)
    except Exception:
        return 0, 0, 0


def count_keywords(text, words):
    text = str(text).lower()
    return sum(1 for w in words if w in text)


def page_text_features(page):
    text = page.get_text("text") or ""
    text_l = text.lower()
    return {
        "page_text_preview": text[:500],
        "flow_word_count": count_keywords(text_l, FLOW_WORDS),
        "table_word_count": count_keywords(text_l, TABLE_WORDS),
        "negative_word_count": count_keywords(text_l, NEGATIVE_VISUAL_WORDS),
        "has_flow_words": count_keywords(text_l, FLOW_WORDS) > 0,
        "has_table_words": count_keywords(text_l, TABLE_WORDS) > 0,
    }


def analyze_pdf_vectors(page):
    drawings = page.get_drawings()
    horizontal_lines = 0
    vertical_lines = 0
    rectangles = 0
    thin_lines = 0

    for d in drawings:
        rect = d.get("rect")
        if rect is None:
            continue
        w, h = float(rect.width), float(rect.height)
        if w > 40 and h < 3:
            horizontal_lines += 1
            thin_lines += 1
        if h > 40 and w < 3:
            vertical_lines += 1
            thin_lines += 1
        if w > 20 and h > 10:
            rectangles += 1

    table_grid_score = 0.0
    if horizontal_lines >= 15:
        table_grid_score += 0.25
    if vertical_lines >= 15:
        table_grid_score += 0.25
    if rectangles >= 35:
        table_grid_score += 0.30
    if thin_lines >= 30:
        table_grid_score += 0.20
    table_grid_score = min(table_grid_score, 1.0)

    return {
        "drawing_count": len(drawings),
        "pdf_horizontal_lines": horizontal_lines,
        "pdf_vertical_lines": vertical_lines,
        "pdf_rectangles": rectangles,
        "pdf_thin_lines": thin_lines,
        "table_grid_score": table_grid_score,
        "is_pdf_table_like": table_grid_score >= 0.70,
    }


In [49]:
# Cell 6: PyMuPDF extraction + candidate normalization
# Every source becomes a candidate image:
# - embedded_image: extracted from PDF as bitmap
# - rendered_vector_page: rendered from PDF vector drawings
# Both types will pass through the same rule + geometry + OpenCLIP pipeline.

doc = pymupdf.open(PDF_PATH)
records = []

for page_index in tqdm(range(len(doc)), desc="Scanning PDF"):
    page = doc[page_index]
    page_number = page_index + 1
    images = page.get_images(full=True)
    vector_metrics = analyze_pdf_vectors(page)
    text_metrics = page_text_features(page)

    # 1) Extract embedded raster images
    for image_index, img in enumerate(images):
        xref = img[0]
        try:
            extracted = doc.extract_image(xref)
            image_ext = extracted.get("ext", "png")
            image_path = EMBEDDED_DIR / f"page_{page_number}_img_{image_index + 1}.{image_ext}"
            with open(image_path, "wb") as f:
                f.write(extracted["image"])
            w, h, area = safe_image_size(image_path)
            records.append({
                "page_number": page_number,
                "source_type": "embedded_image",
                "image_index": image_index + 1,
                "image_path": str(image_path),
                "width": w,
                "height": h,
                "area": area,
                "image_count": len(images),
                **vector_metrics,
                **text_metrics,
            })
        except Exception as e:
            print(f"Embedded image extraction failed on page {page_number}: {e}")

    # 2) Render vector-heavy pages
    if vector_metrics["drawing_count"] >= MIN_DRAWINGS_TO_RENDER:
        # Render to debug-table dir if the page is strongly table-like, otherwise rendered dir.
        output_dir = TABLE_DEBUG_DIR if vector_metrics["is_pdf_table_like"] else RENDERED_DIR
        output_dir.mkdir(parents=True, exist_ok=True)
        suffix = "table_like" if vector_metrics["is_pdf_table_like"] else "rendered"
        render_path = output_dir / f"page_{page_number}_{suffix}.png"
        try:
            pix = page.get_pixmap(matrix=pymupdf.Matrix(RENDER_SCALE, RENDER_SCALE), alpha=False)
            pix.save(str(render_path))
            w, h, area = safe_image_size(render_path)
            records.append({
                "page_number": page_number,
                "source_type": "rendered_vector_page",
                "image_index": 0,
                "image_path": str(render_path),
                "width": w,
                "height": h,
                "area": area,
                "image_count": len(images),
                **vector_metrics,
                **text_metrics,
            })
        except Exception as e:
            print(f"Rendering failed on page {page_number}: {e}")

df_sources = pd.DataFrame(records)
print("Total normalized candidate images:", len(df_sources))
print(df_sources["source_type"].value_counts(dropna=False))
df_sources[["page_number", "source_type", "width", "height", "drawing_count", "table_grid_score", "flow_word_count", "table_word_count"]].head(20)


Scanning PDF: 100%|██████████| 384/384 [00:19<00:00, 19.46it/s]

Total normalized candidate images: 296
source_type
rendered_vector_page    241
embedded_image           55
Name: count, dtype: int64


,page_number,source_type,width,height,drawing_count,table_grid_score,flow_word_count,table_word_count
0,1,embedded_image,2858,4038,90,0.00,0,0
1,1,rendered_vector_page,1191,1684,90,0.00,0,0
2,3,rendered_vector_page,1191,1684,87,0.00,0,0
3,14,rendered_vector_page,1191,1684,721,0.00,2,1
4,15,embedded_image,1646,3263,1,0.00,0,0
5,16,embedded_image,1650,3280,1,0.00,0,0
6,17,embedded_image,1575,2858,1,0.00,0,0
7,18,rendered_vector_page,1191,1684,44,0.70,2,2
8,19,rendered_vector_page,1191,1684,42,0.70,2,0
9,20,rendered_vector_page,1191,1684,37,0.25,1,0


In [50]:
# Cell 7: Unified pre-OpenCLIP rule funnel
# This runs on BOTH embedded images and rendered vector pages.
# Goal: reduce OpenCLIP workload without accidentally dropping likely flowcharts.

def image_geometry_fast(image_path):
    img = cv2.imread(str(image_path))
    if img is None:
        return {
            "img_line_score": 0.0,
            "img_shape_score": 0.0,
            "img_table_score": 0.0,
            "img_horizontal_lines": 0,
            "img_vertical_lines": 0,
            "img_contours": 0,
            "img_rect_like": 0,
            "img_diamond_like": 0,
            "img_arrow_like": 0,
        }

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    binary = cv2.adaptiveThreshold(
        blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 15
    )

    h_img, w_img = gray.shape
    img_area = h_img * w_img

    # Lines
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (45, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 45))
    horizontal = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel)
    vertical = cv2.morphologyEx(binary, cv2.MORPH_OPEN, vertical_kernel)
    h_contours, _ = cv2.findContours(horizontal, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    v_contours, _ = cv2.findContours(vertical, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    horizontal_lines = len(h_contours)
    vertical_lines = len(v_contours)

    # Shapes
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    rect_like = 0
    diamond_like = 0
    ellipse_like = 0
    arrow_like = 0

    for c in contours:
        area = cv2.contourArea(c)
        if area < img_area * 0.0003 or area > img_area * 0.35:
            continue
        peri = cv2.arcLength(c, True)
        if peri <= 0:
            continue
        approx = cv2.approxPolyDP(c, 0.04 * peri, True)
        x, y, w, h = cv2.boundingRect(c)
        aspect = w / max(h, 1)

        if len(approx) == 4:
            if 0.75 <= aspect <= 1.35:
                diamond_like += 1
            else:
                rect_like += 1
        elif len(approx) >= 6 and 0.5 <= aspect <= 2.2:
            ellipse_like += 1

        # Lightweight connector/arrow proxy: small-to-medium elongated components
        if (w > 25 or h > 25) and area < img_area * 0.02:
            arrow_like += 1

    # Scores
    img_line_score = min((horizontal_lines + vertical_lines) / 40, 1.0)
    img_shape_score = min((rect_like * 0.08) + (diamond_like * 0.20) + (ellipse_like * 0.12) + (arrow_like * 0.03), 1.0)

    # A table tends to have many horizontal + vertical lines and very few diamond/ellipse signals.
    img_table_score = 0.0
    if horizontal_lines >= 12:
        img_table_score += 0.30
    if vertical_lines >= 12:
        img_table_score += 0.30
    if rect_like >= 30:
        img_table_score += 0.25
    if diamond_like == 0 and ellipse_like == 0:
        img_table_score += 0.15
    img_table_score = min(img_table_score, 1.0)

    return {
        "img_line_score": img_line_score,
        "img_shape_score": img_shape_score,
        "img_table_score": img_table_score,
        "img_horizontal_lines": horizontal_lines,
        "img_vertical_lines": vertical_lines,
        "img_contours": len(contours),
        "img_rect_like": rect_like,
        "img_diamond_like": diamond_like,
        "img_ellipse_like": ellipse_like,
        "img_arrow_like": arrow_like,
    }

# Apply fast image geometry to every normalized candidate image
fast_records = []
for _, row in tqdm(df_sources.iterrows(), total=len(df_sources), desc="Fast image geometry"):
    fast_records.append({**row.to_dict(), **image_geometry_fast(row["image_path"])})

df_fast = pd.DataFrame(fast_records)


def pre_rule_decision(row):
    # Hard rejects
    if row["width"] < MIN_WIDTH or row["height"] < MIN_HEIGHT or row["area"] < MIN_AREA:
        return "reject_small"

    if row.get("negative_word_count", 0) >= 2 and row.get("flow_word_count", 0) == 0:
        return "reject_cover_or_logo_words"

    # Strong table/map reject: PDF-grid OR image-grid + table words + no flow words
    strong_table = max(row.get("table_grid_score", 0), row.get("img_table_score", 0)) >= 0.85
    table_text = row.get("table_word_count", 0) >= 2
    flow_text = row.get("flow_word_count", 0) >= 1
    has_flow_shapes = row.get("img_diamond_like", 0) >= 1 or row.get("img_ellipse_like", 0) >= 1

    if strong_table and table_text and not flow_text:
        return "reject_strong_table_text"

    if strong_table and not has_flow_shapes and row.get("img_arrow_like", 0) < 3:
        return "reject_strong_table_grid"

    # Ranking score before OpenCLIP. This helps reduce OpenCLIP inputs.
    score = 0.0
    score += 0.35 * row.get("img_shape_score", 0)
    score += 0.20 * row.get("img_line_score", 0)
    score += 0.20 if row.get("flow_word_count", 0) > 0 else 0.0
    score += 0.15 if row.get("img_diamond_like", 0) > 0 else 0.0
    score += 0.10 if row.get("img_arrow_like", 0) >= 3 else 0.0
    score -= 0.25 * max(row.get("table_grid_score", 0), row.get("img_table_score", 0))
    score -= 0.10 if row.get("table_word_count", 0) > row.get("flow_word_count", 0) else 0.0

    return "send_to_openclip" if score >= 0.10 else "reject_low_signal"


df_fast["pre_rule_decision"] = df_fast.apply(pre_rule_decision, axis=1)

# Only send high-ranked items to OpenCLIP. This prevents hundreds of pages going to OpenCLIP.
df_pre_rejected = df_fast[df_fast["pre_rule_decision"] != "send_to_openclip"].copy()
df_openclip_pool = df_fast[df_fast["pre_rule_decision"] == "send_to_openclip"].copy()

# Candidate priority score for limiting OpenCLIP workload.
df_openclip_pool["preclip_candidate_score"] = (
    0.35 * df_openclip_pool["img_shape_score"] +
    0.20 * df_openclip_pool["img_line_score"] +
    0.20 * (df_openclip_pool["flow_word_count"] > 0).astype(float) +
    0.15 * (df_openclip_pool["img_diamond_like"] > 0).astype(float) +
    0.10 * (df_openclip_pool["img_arrow_like"] >= 3).astype(float) -
    0.25 * df_openclip_pool[["table_grid_score", "img_table_score"]].max(axis=1)
)

df_clip_input = df_openclip_pool.sort_values("preclip_candidate_score", ascending=False).head(MAX_OPENCLIP_INPUTS).copy()
df_pre_rejected = pd.concat([
    df_pre_rejected,
    df_openclip_pool.drop(df_clip_input.index).assign(pre_rule_decision="defer_low_rank_before_openclip")
], ignore_index=True)

print("Total normalized candidates:", len(df_fast))
print("Pre-rejected/deferred:", len(df_pre_rejected))
print("Sent to OpenCLIP:", len(df_clip_input))
print(df_fast["pre_rule_decision"].value_counts())

df_clip_input[["page_number", "source_type", "preclip_candidate_score", "pre_rule_decision", "img_shape_score", "img_table_score", "table_grid_score", "flow_word_count", "table_word_count"]].head(20)


Fast image geometry: 100%|██████████| 296/296 [00:11<00:00, 25.88it/s]

Total normalized candidates: 296
Pre-rejected/deferred: 196
Sent to OpenCLIP: 100
pre_rule_decision
send_to_openclip              224
reject_small                   35
reject_cover_or_logo_words     24
reject_low_signal              13
Name: count, dtype: int64


,page_number,source_type,preclip_candidate_score,pre_rule_decision,img_shape_score,img_table_score,table_grid_score,flow_word_count,table_word_count
279,357,rendered_vector_page,0.925,send_to_openclip,1.0,0.3,0.25,1,3
269,339,embedded_image,0.925,send_to_openclip,1.0,0.3,0.00,2,1
3,14,rendered_vector_page,0.850,send_to_openclip,1.0,0.6,0.00,2,1
117,138,rendered_vector_page,0.850,send_to_openclip,1.0,0.6,0.45,1,8
140,177,rendered_vector_page,0.850,send_to_openclip,1.0,0.6,0.45,1,6
125,153,rendered_vector_page,0.850,send_to_openclip,1.0,0.6,0.45,1,6
134,168,rendered_vector_page,0.850,send_to_openclip,1.0,0.6,0.45,1,6
133,167,rendered_vector_page,0.850,send_to_openclip,1.0,0.6,0.45,2,6
145,182,rendered_vector_page,0.850,send_to_openclip,1.0,0.6,0.45,1,6
154,194,rendered_vector_page,0.850,send_to_openclip,1.0,0.6,0.45,1,6


In [51]:
# Cell 8: Load Paddle layout detector
# This stage runs BEFORE OpenCLIP and removes obvious document-layout cases.

layout_pipeline = None

if USE_PADDLE_LAYOUT:
    try:
        from paddlex import create_pipeline
        layout_pipeline = create_pipeline(pipeline="layout_parsing")
        print("Paddle layout detector loaded successfully.")
    except Exception as e:
        print("Paddle layout detector unavailable. Continuing without it.")
        print("Reason:", e)
else:
    print("Skipping Paddle layout detector.")


Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None, None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/UVDoc`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('RT-DETR-H_layout_17cls', None, None)
Using official model (RT-DETR-H_layout_17cls), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/RT-DETR-H_layout_17cls`.


...

Creating model: ('PP-OCRv4_server_seal_det', None, None)
Using official model (PP-OCRv4_server_seal_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv4_server_seal_det`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-OCRv4_server_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv4_server_rec`.
Creating model: ('SLANet_plus', None, None)
Using official model (SLANet_plus), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/SLANet_plus`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Paddle layout detector loaded successfully.


In [52]:
# Cell 9: Paddle layout detection before OpenCLIP
# Goal: reduce OpenCLIP workload by removing obvious table/text/reference-map pages.

def _recursive_text_blob(obj):
    parts = []
    if isinstance(obj, dict):
        for v in obj.values():
            parts.append(_recursive_text_blob(v))
    elif isinstance(obj, list):
        for v in obj:
            parts.append(_recursive_text_blob(v))
    elif obj is None:
        pass
    else:
        parts.append(str(obj))
    return " ".join([p for p in parts if p])

LAYOUT_TABLE_WORDS = [
    "table", "cell", "row", "column", "header", "reference map",
    "article numbers", "annex numbers"
]
LAYOUT_TEXT_WORDS = [
    "text", "title", "header", "footer", "paragraph", "caption"
]
LAYOUT_FIGURE_WORDS = [
    "figure", "image", "diagram", "chart", "graphic", "illustration"
]

def run_layout_detection(image_path):
    if layout_pipeline is None:
        return {
            "layout_table_signal": 0,
            "layout_text_signal": 0,
            "layout_figure_signal": 0,
            "layout_text_preview": "",
        }

    try:
        raw = layout_pipeline.predict(str(image_path))
        text_blob = _recursive_text_blob(raw).lower()
    except Exception as e:
        text_blob = f"layout_error: {e}".lower()

    return {
        "layout_table_signal": count_keywords(text_blob, LAYOUT_TABLE_WORDS),
        "layout_text_signal": count_keywords(text_blob, LAYOUT_TEXT_WORDS),
        "layout_figure_signal": count_keywords(text_blob, LAYOUT_FIGURE_WORDS),
        "layout_text_preview": text_blob[:500],
    }

# Only stronger rule-pass items go to layout detection.
df_layout_input = df_clip_input.copy()

if len(df_layout_input) > MAX_LAYOUT_INPUTS:
    # Prioritize pages that already look more flow-like and less table-like.
    df_layout_input["layout_priority_score"] = (
        0.30 * df_layout_input["preclip_candidate_score"]
        + 0.15 * df_layout_input["img_shape_score"]
        + 0.10 * df_layout_input["flow_word_count"]
        - 0.20 * df_layout_input["table_grid_score"]
        - 0.15 * df_layout_input["img_table_score"]
        - 0.10 * df_layout_input["table_word_count"]
    )
    df_layout_input = df_layout_input.sort_values("layout_priority_score", ascending=False).head(MAX_LAYOUT_INPUTS).copy()

layout_records = []
for _, row in tqdm(df_layout_input.iterrows(), total=len(df_layout_input), desc="Paddle layout detection"):
    metrics = run_layout_detection(row["image_path"])
    layout_records.append({
        "image_path": row["image_path"],
        **metrics,
    })

df_layout = pd.DataFrame(layout_records)

if df_layout.empty:
    df_after_layout = df_clip_input.copy()
    df_after_layout["layout_table_signal"] = 0
    df_after_layout["layout_text_signal"] = 0
    df_after_layout["layout_figure_signal"] = 0
    df_after_layout["layout_text_preview"] = ""
    df_after_layout["layout_decision"] = "send_to_openclip"
else:
    df_after_layout = df_clip_input.merge(df_layout, on="image_path", how="left")
    for col in ["layout_table_signal", "layout_text_signal", "layout_figure_signal"]:
        df_after_layout[col] = df_after_layout[col].fillna(0)

    def layout_decision(row):
        # Strong table/reference-map style layout
        if (
            row.get("layout_table_signal", 0) >= 2
            and row.get("layout_figure_signal", 0) == 0
            and row.get("flow_word_count", 0) <= 1
        ):
            return "reject_layout_table"

        # Large text-like document page / cover / contents page
        if (
            row.get("layout_text_signal", 0) >= 3
            and row.get("layout_figure_signal", 0) == 0
            and row.get("negative_word_count", 0) > 0
        ):
            return "reject_layout_text"

        # If it has some figure/diagram signal or already looks flow-like, keep it.
        return "send_to_openclip"

    df_after_layout["layout_decision"] = df_after_layout.apply(layout_decision, axis=1)

print("Layout stage decisions:")
print(df_after_layout["layout_decision"].value_counts(dropna=False))

df_openclip_input = df_after_layout[df_after_layout["layout_decision"] == "send_to_openclip"].copy()

if len(df_openclip_input) > MAX_OPENCLIP_INPUTS:
    df_openclip_input["openclip_priority_score"] = (
        0.35 * df_openclip_input["preclip_candidate_score"]
        + 0.20 * df_openclip_input["img_shape_score"]
        + 0.10 * df_openclip_input["flow_word_count"]
        + 0.05 * (df_openclip_input["layout_figure_signal"] > 0).astype(float)
        - 0.25 * df_openclip_input["table_grid_score"]
        - 0.20 * df_openclip_input["img_table_score"]
        - 0.20 * (df_openclip_input["layout_table_signal"] >= 2).astype(float)
        - 0.10 * df_openclip_input["table_word_count"]
    )
    df_openclip_input = df_openclip_input.sort_values("openclip_priority_score", ascending=False).head(MAX_OPENCLIP_INPUTS).copy()

print("Sent to OpenCLIP after layout stage:", len(df_openclip_input))

df_after_layout[[
    "page_number", "source_type", "preclip_candidate_score", "pre_rule_decision",
    "layout_table_signal", "layout_text_signal", "layout_figure_signal", "layout_decision"
]].head(30)



Paddle layout detection: 100%|██████████| 100/100 [00:00<00:00, 14186.24it/s]

Layout stage decisions:
layout_decision
send_to_openclip    100
Name: count, dtype: int64
Sent to OpenCLIP after layout stage: 100


,page_number,source_type,preclip_candidate_score,pre_rule_decision,layout_table_signal,layout_text_signal,layout_figure_signal,layout_decision
0,357,rendered_vector_page,0.925,send_to_openclip,0,0,0,send_to_openclip
1,339,embedded_image,0.925,send_to_openclip,0,0,0,send_to_openclip
2,14,rendered_vector_page,0.850,send_to_openclip,0,0,0,send_to_openclip
3,138,rendered_vector_page,0.850,send_to_openclip,0,0,0,send_to_openclip
4,177,rendered_vector_page,0.850,send_to_openclip,0,0,0,send_to_openclip
5,153,rendered_vector_page,0.850,send_to_openclip,0,0,0,send_to_openclip
6,168,rendered_vector_page,0.850,send_to_openclip,0,0,0,send_to_openclip
7,167,rendered_vector_page,0.850,send_to_openclip,0,0,0,send_to_openclip
8,182,rendered_vector_page,0.850,send_to_openclip,0,0,0,send_to_openclip
9,194,rendered_vector_page,0.850,send_to_openclip,0,0,0,send_to_openclip


In [53]:
# Cell 10: Load OpenCLIP
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

OPENCLIP_MODEL = "ViT-B-32"
OPENCLIP_PRETRAINED = "laion2b_s34b_b79k"

model, _, preprocess = open_clip.create_model_and_transforms(
    OPENCLIP_MODEL,
    pretrained=OPENCLIP_PRETRAINED,
    device=DEVICE
)
tokenizer = open_clip.get_tokenizer(OPENCLIP_MODEL)
model.eval()

LABELS = [
    "a flowchart diagram with boxes arrows decisions yes and no branches",
    "a workflow or process diagram with connected steps",
    "a decision tree or branching process diagram",
    "a regulatory process flow diagram with connected boxes and arrows",
    "a data table with rows columns and grid cells",
    "a reference map table with article numbers and annex numbers",
    "a logo cover page banner or decorative image",
    "a normal document page with text paragraphs",
    "an organization chart or hierarchy diagram",
    "a graph chart or plot",
]

LABEL_FLOWCHART = LABELS[0]
LABEL_WORKFLOW = LABELS[1]
LABEL_DECISION_TREE = LABELS[2]
LABEL_REGULATORY_FLOW = LABELS[3]
LABEL_TABLE = LABELS[4]
LABEL_REFERENCE_MAP = LABELS[5]
LABEL_LOGO = LABELS[6]
LABEL_TEXT_PAGE = LABELS[7]
LABEL_ORG_CHART = LABELS[8]
LABEL_GRAPH = LABELS[9]

text_tokens = tokenizer(LABELS).to(DEVICE)
with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)

print("OpenCLIP loaded with", len(LABELS), "labels")


Device: cpu
OpenCLIP loaded with 10 labels


In [54]:
# Cell 11: Run OpenCLIP only on post-layout optimized candidate set

def classify_openclip(image_path):
    image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)[0]

    scores = {LABELS[i]: float(probs[i]) for i in range(len(LABELS))}
    best_label = max(scores, key=scores.get)
    return best_label, scores[best_label], scores

clip_records = []
for _, row in tqdm(df_openclip_input.iterrows(), total=len(df_openclip_input), desc="OpenCLIP classification"):
    best_label, best_score, scores = classify_openclip(row["image_path"])
    clip_records.append({
        **row.to_dict(),
        "clip_best_label": best_label,
        "clip_best_score": best_score,
        "clip_flowchart_score": scores[LABEL_FLOWCHART],
        "clip_workflow_score": scores[LABEL_WORKFLOW],
        "clip_decision_tree_score": scores[LABEL_DECISION_TREE],
        "clip_regulatory_flow_score": scores[LABEL_REGULATORY_FLOW],
        "clip_table_score": scores[LABEL_TABLE],
        "clip_reference_map_score": scores[LABEL_REFERENCE_MAP],
        "clip_logo_score": scores[LABEL_LOGO],
        "clip_text_page_score": scores[LABEL_TEXT_PAGE],
        "clip_org_chart_score": scores[LABEL_ORG_CHART],
        "clip_graph_score": scores[LABEL_GRAPH],
    })

df_clip = pd.DataFrame(clip_records)
print("OpenCLIP completed:", len(df_clip))
df_clip[[
    "page_number", "source_type", "clip_best_label", "clip_best_score",
    "clip_flowchart_score", "clip_workflow_score", "clip_regulatory_flow_score",
    "clip_table_score", "clip_reference_map_score",
    "img_shape_score", "img_table_score", "layout_table_signal"
]].head(20)


OpenCLIP classification: 100%|██████████| 100/100 [00:17<00:00,  5.70it/s]


OpenCLIP completed: 100


,page_number,source_type,clip_best_label,clip_best_score,clip_flowchart_score,clip_workflow_score,clip_regulatory_flow_score,clip_table_score,clip_reference_map_score,img_shape_score,img_table_score,layout_table_signal
0,357,rendered_vector_page,a normal document page with text paragraphs,0.999646,2.591732e-07,1.202811e-04,6.473373e-05,0.000059,0.000058,1.0,0.3,0
1,339,embedded_image,a data table with rows columns and grid cells,0.396180,7.644335e-05,4.408986e-02,1.634226e-02,0.396180,0.037125,1.0,0.3,0
2,14,rendered_vector_page,a regulatory process flow diagram with connect...,0.820295,4.609833e-03,1.296792e-01,8.202954e-01,0.000001,0.000073,1.0,0.6,0
3,138,rendered_vector_page,a reference map table with article numbers and...,0.898365,1.676599e-05,2.414346e-05,4.222853e-04,0.045668,0.898365,1.0,0.6,0
4,177,rendered_vector_page,a reference map table with article numbers and...,0.627847,4.870610e-05,4.081462e-06,1.034707e-05,0.341392,0.627847,1.0,0.6,0
5,153,rendered_vector_page,a reference map table with article numbers and...,0.940521,1.194300e-06,8.938919e-08,5.402992e-08,0.022955,0.940521,1.0,0.6,0
6,168,rendered_vector_page,a reference map table with article numbers and...,0.532458,3.687157e-07,3.934599e-06,3.047634e-06,0.072567,0.532458,1.0,0.6,0
7,167,rendered_vector_page,a reference map table with article numbers and...,0.579748,2.754596e-05,1.121355e-05,1.531706e-04,0.159640,0.579748,1.0,0.6,0
8,182,rendered_vector_page,a reference map table with article numbers and...,0.930154,4.800491e-07,2.304049e-06,9.025676e-07,0.067655,0.930154,1.0,0.6,0
9,194,rendered_vector_page,a data table with rows columns and grid cells,0.498334,3.555329e-04,4.378296e-04,1.694560e-03,0.498334,0.187749,1.0,0.6,0


In [55]:
# Cell 12: Score fusion after OpenCLIP
# Uses the same image-geometry metrics for embedded images and rendered vector pages.

def score_fusion(row):
    semantic_flow = max(
        row.get("clip_flowchart_score", 0),
        row.get("clip_workflow_score", 0),
        row.get("clip_decision_tree_score", 0),
        row.get("clip_regulatory_flow_score", 0),
    )
    semantic_table = max(
        row.get("clip_table_score", 0),
        row.get("clip_reference_map_score", 0),
    )
    semantic_negative = max(
        row.get("clip_logo_score", 0),
        row.get("clip_text_page_score", 0),
        row.get("clip_graph_score", 0),
    )
    geometry = row.get("img_shape_score", 0)
    table_grid = max(row.get("table_grid_score", 0), row.get("img_table_score", 0))

    score = 0.45 * semantic_flow + 0.30 * geometry
    score += 0.10 if row.get("flow_word_count", 0) > 0 else 0.0
    score += 0.08 if row.get("img_diamond_like", 0) > 0 else 0.0
    score += 0.05 if row.get("img_arrow_like", 0) >= 3 else 0.0
    score += 0.05 if row.get("layout_figure_signal", 0) > 0 else 0.0
    score -= 0.30 * table_grid
    score -= 0.22 * semantic_table
    score -= 0.12 * semantic_negative
    score -= 0.10 if row.get("layout_table_signal", 0) >= 2 else 0.0
    score -= 0.10 if row.get("table_word_count", 0) > row.get("flow_word_count", 0) else 0.0
    return float(max(0, min(score, 1.0)))

def first_pass_decision(row):
    table_grid = max(row.get("table_grid_score", 0), row.get("img_table_score", 0))
    table_semantic = max(row.get("clip_table_score", 0), row.get("clip_reference_map_score", 0))
    flow_semantic = max(
        row.get("clip_flowchart_score", 0),
        row.get("clip_workflow_score", 0),
        row.get("clip_decision_tree_score", 0),
        row.get("clip_regulatory_flow_score", 0),
    )

    # Hard table veto if layout + structure + semantic agree.
    if (
        (table_grid >= 0.75 or row.get("layout_table_signal", 0) >= 2)
        and table_semantic > flow_semantic
        and row.get("layout_figure_signal", 0) == 0
    ):
        return "rejected_table"

    score = row.get("first_pass_score", 0)
    if score >= 0.56:
        return "flowchart_candidate"
    if score >= 0.30:
        return "needs_paddle_fallback"
    return "rejected_other"

if df_clip.empty:
    df_geo = pd.DataFrame()
    needs_paddle = pd.DataFrame()
else:
    df_geo = df_clip.copy()
    df_geo["first_pass_score"] = df_geo.apply(score_fusion, axis=1)
    df_geo["first_pass_decision"] = df_geo.apply(first_pass_decision, axis=1)
    needs_paddle = (
        df_geo[df_geo["first_pass_decision"] == "needs_paddle_fallback"]
        .sort_values("first_pass_score", ascending=False)
        .head(MAX_PADDLEOCR_VL_ITEMS)
        .copy()
    )

print("First pass decisions:")
print(df_geo["first_pass_decision"].value_counts(dropna=False) if not df_geo.empty else "No OpenCLIP results")
print("Needs PaddleOCR-VL fallback:", len(needs_paddle))

df_geo[[
    "page_number", "source_type", "first_pass_score", "first_pass_decision",
    "clip_best_label", "img_shape_score", "img_table_score", "table_grid_score",
    "layout_table_signal", "layout_figure_signal", "flow_word_count", "table_word_count"
]].head(30) if not df_geo.empty else df_geo


First pass decisions:
first_pass_decision
rejected_other           84
rejected_table           10
needs_paddle_fallback     4
flowchart_candidate       2
Name: count, dtype: int64
Needs PaddleOCR-VL fallback: 4


,page_number,source_type,first_pass_score,first_pass_decision,clip_best_label,img_shape_score,img_table_score,table_grid_score,layout_table_signal,layout_figure_signal,flow_word_count,table_word_count
0,357,rendered_vector_page,0.220084,rejected_other,a normal document page with text paragraphs,1.0,0.3,0.25,0,0,1,3
1,339,embedded_image,0.332683,needs_paddle_fallback,a data table with rows columns and grid cells,1.0,0.3,0.00,0,0,2,1
2,14,rendered_vector_page,0.719105,flowchart_candidate,a regulatory process flow diagram with connect...,1.0,0.6,0.00,0,0,2,1
3,138,rendered_vector_page,0.046673,rejected_other,a reference map table with article numbers and...,1.0,0.6,0.45,0,0,1,8
4,177,rendered_vector_page,0.108656,rejected_other,a reference map table with article numbers and...,1.0,0.6,0.45,0,0,1,6
5,153,rendered_vector_page,0.040358,rejected_other,a reference map table with article numbers and...,1.0,0.6,0.45,0,0,1,6
6,168,rendered_vector_page,0.085526,rejected_other,a reference map table with article numbers and...,1.0,0.6,0.45,0,0,1,6
7,167,rendered_vector_page,0.091449,rejected_other,a reference map table with article numbers and...,1.0,0.6,0.45,0,0,2,6
8,182,rendered_vector_page,0.045240,rejected_other,a reference map table with article numbers and...,1.0,0.6,0.45,0,0,1,6
9,194,rendered_vector_page,0.106641,rejected_other,a data table with rows columns and grid cells,1.0,0.6,0.45,0,0,1,6


In [ ]:
# Cell 13: PaddleOCR-VL fallback only for uncertain pages
# This is intentionally limited. It should NEVER run on every page.

paddle_results = []

if USE_PADDLEOCR_VL and len(needs_paddle) > 0:
    try:
        from paddleocr import PaddleOCRVL
        pipeline = PaddleOCRVL(pipeline_version="v1.6")

        for _, row in tqdm(needs_paddle.iterrows(), total=len(needs_paddle), desc="PaddleOCR-VL fallback"):
            try:
                output = pipeline.predict(row["image_path"])
                pieces = []
                for res in output:
                    pieces.append(str(res)[:3000])
                text_blob = " ".join(pieces).lower()
            except Exception as e:
                text_blob = f"paddle_error: {e}"

            paddle_results.append({
                "page_number": row["page_number"],
                "image_path": row["image_path"],
                "paddle_flow_signal": count_keywords(text_blob, FLOW_WORDS),
                "paddle_table_signal": count_keywords(text_blob, TABLE_WORDS),
                "paddle_negative_signal": count_keywords(text_blob, NEGATIVE_VISUAL_WORDS),
                "paddle_text_preview": text_blob[:500],
            })

    except Exception as e:
        print("PaddleOCR-VL unavailable or failed to load:", e)
else:
    print("Skipping PaddleOCR-VL fallback.")

df_paddle = pd.DataFrame(paddle_results)
print("Paddle fallback completed:", len(df_paddle))
df_paddle.head(20)


Creating model: ('PP-DocLayoutV3', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-DocLayoutV3`.
Creating model: ('PaddleOCR-VL-1.6-0.9B', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PaddleOCR-VL-1.6`.
Bucketed engine_config has no entry for resolved engine 'paddle_dynamic'; using an empty config for that engine.
Loading configuration file /root/.paddlex/official_models/PaddleOCR-VL-1.6/config.json
Loading weights file /root/.paddlex/official_models/PaddleOCR-VL-1.6/model.safetensors
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16- num_key_value_heads: 2
use GQA - num_heads: 16-

In [ ]:
# Cell 14: Final decision after fallback
# Final output stays in dataframe and display; no export folders / CSV.

if df_geo.empty:
    df_final = pd.DataFrame()
else:
    df_final = df_geo.copy()

    if not df_paddle.empty:
        df_final = df_final.merge(
            df_paddle[["image_path", "paddle_flow_signal", "paddle_table_signal", "paddle_negative_signal", "paddle_text_preview"]],
            on="image_path",
            how="left"
        )
    else:
        df_final["paddle_flow_signal"] = 0
        df_final["paddle_table_signal"] = 0
        df_final["paddle_negative_signal"] = 0
        df_final["paddle_text_preview"] = ""

    df_final[["paddle_flow_signal", "paddle_table_signal", "paddle_negative_signal"]] = df_final[["paddle_flow_signal", "paddle_table_signal", "paddle_negative_signal"]].fillna(0)

    def final_decision(row):
        score = row["first_pass_score"]

        # Paddle fallback adjustment for uncertain cases only.
        if row["first_pass_decision"] == "needs_paddle_fallback":
            score += 0.08 * min(row.get("paddle_flow_signal", 0), 3)
            score -= 0.10 * min(row.get("paddle_table_signal", 0), 3)
            score -= 0.06 * min(row.get("paddle_negative_signal", 0), 2)

        # Strong table veto remains final.
        table_grid = max(row.get("table_grid_score", 0), row.get("img_table_score", 0))
        table_semantic = max(row.get("clip_table_score", 0), row.get("clip_reference_map_score", 0))
        flow_semantic = max(row.get("clip_flowchart_score", 0), row.get("clip_workflow_score", 0), row.get("clip_decision_tree_score", 0))

        if table_grid >= 0.85 and table_semantic >= flow_semantic:
            return "rejected_table", max(0, min(score, 1))

        final_score = max(0, min(score, 1))
        if final_score >= 0.58:
            return "flowchart_candidate", final_score
        if final_score >= 0.38:
            return "needs_review"
        return "rejected_other", final_score

    decisions = df_final.apply(final_decision, axis=1)
    df_final["final_decision"] = [d[0] for d in decisions]
    df_final["final_score"] = [d[1] for d in decisions]

print("Final decisions:")
print(df_final["final_decision"].value_counts(dropna=False) if not df_final.empty else "No final results")

df_final[["page_number", "source_type", "final_decision", "final_score", "first_pass_score", "clip_best_label", "img_shape_score", "img_table_score", "table_grid_score", "paddle_flow_signal", "paddle_table_signal"]].head(30) if not df_final.empty else df_final


In [ ]:
# Cell 15: Display flowchart candidates
# Displays only final flowchart candidates. Increase DISPLAY_LIMIT if needed.

DISPLAY_LIMIT = 40

if df_final.empty:
    print("No final results available.")
else:
    flowcharts = df_final[df_final["final_decision"] == "flowchart_candidate"].sort_values("final_score", ascending=False).copy()
    print("Flowchart candidates:", len(flowcharts))

    display(flowcharts[[
        "page_number", "source_type", "final_score", "first_pass_score",
        "clip_best_label", "clip_flowchart_score", "clip_workflow_score", "clip_table_score",
        "img_shape_score", "img_table_score", "table_grid_score", "flow_word_count", "table_word_count"
    ]].head(DISPLAY_LIMIT))

    for _, row in flowcharts.head(DISPLAY_LIMIT).iterrows():
        print("=" * 80)
        print(f"Page: {row['page_number']} | Source: {row['source_type']} | Final score: {row['final_score']:.3f}")
        print(f"Image: {row['image_path']}")
        display(Image.open(row["image_path"]))
